# 02 · Science suite

Survey context, star/galaxy split, classifier validation, label evolution, variability, distances, data quality and
the solar-system population. Every figure and table comes from the functions in `scripts/alert_science_suite.py`,
which also made the delivered figures.

| Code | Figure | Question |
|---|---|---|
| S1–S3 | sky density, alerts per night, programme composition | Where and when do the alerts come from? |
| C1 | template colours split by Legacy DR8 morphology | Which alerting objects are stars and which galaxies? |
| K1–K2 | Fink labels vs TNS / VSX / SIMBAD; SuperNNova ROC | How reliable are Fink's labels? |
| L1 | label transitions vs detection index | How stable are labels as detections accumulate? |
| V1–V2 | variability amplitude; colour space coloured by skewness | What variability do we see? |
| D1–D2 | Gaia HR diagram; brightest absolute magnitude vs redshift | Distances and luminosities |
| Q1 | reliability by band and brightness | Data quality |
| O1 | SSO share vs ecliptic latitude; SSO magnitudes | Solar-system alerts (kept separate) |

**Inputs:** the catalog and its pinned raw files, plus notebook 01's
`01_template_colours/tables/dia_object_template_photometry.parquet`.
**Outputs:** `RUN_ROOT/02_science_suite/` holding `figures/`, `tables/` (including the per-object
`dia_object_science_table.parquet` used by notebooks 03–05) and `summary.json`.

Run time on Arnor: about 8 minutes. Reading 7.1 M alerts needs roughly 10 GB of memory.

In [ ]:
# Shared configuration: paths, run folder, helpers (edit notebooks/science/_setup.py, not this cell).
from _setup import *
describe_config()
import alert_science_suite as suite
template_photometry = require(RUN_ROOT / "01_template_colours/tables/dia_object_template_photometry.parquet", "01_template_colours")
out = step_dir("02_science_suite")

import matplotlib
matplotlib.use("Agg")                 # figures are saved to files and displayed with show()
import matplotlib.pyplot as plt
suite.style(plt)
figs, tabs = [], {}                   # figure records and result tables, filled by each analysis below


def new_figures(before):
    """Show the figures written since `before` (the length of `figs` before an analysis ran)."""
    show([out / f["file"] for f in figs[before:]])

## Step 1 · Load every alert and build the object table
All delivered alerts (DIA and SSO) are read from the catalog's pinned Parquet files. Populations follow
`analysis_contract_v1` and are asserted against the catalog's totals. The object table holds each DIA object's latest
Fink labels and scores, its latest valid crossmatch values (SIMBAD, Gaia, Legacy, VSX, TNS) and its brightest delivered
difference flux.

In [ ]:
files, qc, req = suite.load_catalog_inputs(CATALOG, DATA_ROOT)
src, counts = suite.prepare_sources(files, qc)
dia, obj = suite.build_object_table(src, template_photometry)
print("alerts:", f"{len(src):,}", counts, "   DIA objects:", f"{len(obj):,}")

## S · Survey context (S1 sky density, S2 alerts per night, S3 programme composition; tables T1, T2)

In [ ]:
n = len(figs); suite.survey(src, req, out, plt, figs, tabs); new_figures(n)
tabs["T1"]

## C · Star/galaxy split with Legacy DR8 `pstar` (C1)
Extended objects (pstar < 0.5) are red contours and point-like objects are blue dots. The top-left panel replaces
r PSF − CModel, which the alerts cannot provide.

In [ ]:
n = len(figs); suite.star_galaxy(obj, out, plt, figs, tabs); new_figures(n)
tabs["C_counts"]

## K · Classifier validation (K1 confusion matrix and recall vs detections; K2 SuperNNova ROC)
External labels come from Fink's own crossmatches, with precedence TNS (spectroscopic) > VSX > SIMBAD. Each
group's expected CATS class is SN → SN-like, SLSN/TDE → Long, CV → Fast, eclipsing/pulsating/rotational →
Periodic, and QSO/AGN → non-Periodic. Intervals are 95% Wilson. The AUC uses 1,000 bootstrap resamples with a fixed
seed.

In [ ]:
n = len(figs); suite.classifier_validation(obj, out, plt, figs, tabs); new_figures(n)
display(tabs["K2"]); tabs["K_roc"]

## L · Label evolution (L1)
k counts detections inside this cohort. Fink's classifier also sees earlier Rubin history.

In [ ]:
n = len(figs); suite.label_evolution(dia, out, plt, figs, tabs); new_figures(n)
tabs["L"]

## V · Variability (V1 amplitude vs template magnitude, V2 colour space coloured by skewness)
Uses Fink's SNAD features on the i-band difference flux at each object's latest i-band alert, for objects with
≥ 5 delivered i detections. Only detected epochs enter, so amplitudes are biased high near the limit.

In [ ]:
n = len(figs); suite.variability(obj, files, out, plt, figs, tabs); new_figures(n)
tabs["V"]

## D · Distances (D1 Gaia-parallax HR diagram, D2 brightest delivered absolute magnitude vs redshift)
D2 uses flat ΛCDM with Planck18 H0 and Ωm, integrated numerically. There is no K or extinction correction, and the
peaks are lower limits.

In [ ]:
n = len(figs); suite.distances(obj, out, plt, figs, tabs); new_figures(n)
tabs["D_gaia"], tabs["D_peak"]

## Q · Data quality (Q1)

In [ ]:
n = len(figs); suite.quality(src, obj, out, plt, figs, tabs); new_figures(n)
tabs["Q"]

## O · Solar-system objects (O1), kept separate per `analysis_contract_v1`

In [ ]:
n = len(figs); suite.sso(src, out, plt, figs, tabs); new_figures(n)
tabs["O"]

## Save the object table and the summary

In [ ]:
obj.drop(columns=["latest_sid_i"]).to_parquet(out / "tables" / "dia_object_science_table.parquet", index=False)
summary = suite.build_summary(out, code_version(), CATALOG, template_photometry, files, counts, obj, figs, tabs, {},
                              round(time.time() - T0, 1))
save_json(suite.jsonable(summary), out / "summary.json")
print(len(figs), "figures written to", out / "figures")